⏺ COO stands for Coordinate format — it's a way to represent a sparse matrix as three parallel
  arrays:                                                                                          
                                    
  ┌───────┬─────────────────────────────────────┐
  │ array │               meaning               │                                                    
  ├───────┼─────────────────────────────────────┤
  │ rows  │ row index of each non-zero entry    │                                                    
  ├───────┼─────────────────────────────────────┤
  │ cols  │ column index of each non-zero entry │                                                    
  ├───────┼─────────────────────────────────────┤
  │ data  │ value at that (row, col) position   │                                                    
  └───────┴─────────────────────────────────────┘                                                    
  
  So for each edge src -> dst, it stores:                                                            
                  
  A[dst, src] = 1 / out_degree[src]                                                                  
                                                                                                     
  meaning "what fraction of src's rank flows to dst". The matrix is column-stochastic: each column   
  src sums to 1 because all of src's outgoing edges share its weight equally.                        
                                                                                                     
  SciPy accepts this COO format directly and converts it to CSR internally:                          
  sp.csr_matrix((data, (rows, cols)), shape=(N, N))
                                                                                                     
  CSR (Compressed Sparse Row) is then efficient for the matrix-vector multiply A @ v used in PageRank
   power iteration.     

In [ ]:
from loader import WebGraphLoader                                                     
from core import PageRankEngine  
from analytical import AnalyticalPageRank

In [ ]:
loader = WebGraphLoader("../../data/toy-dataset.txt").load()  
print(loader.n_nodes)
print(loader.n_edges)
print(loader.A.shape == (loader.n_nodes,loader.n_nodes))
print(loader.dangling_mask.shape)
loader.summary()
print(loader.idx_to_id)

In [ ]:

engine = PageRankEngine(loader.A, loader.dangling_mask, p=0.15, tol=1e-8)        
result_iterative = engine.run()        
r_analytical = AnalyticalPageRank(loader.A, loader.dangling_mask, p=0.15)
result_analytical = r_analytical.compute()                                                                        

In [4]:
import numpy as np
print(np.testing.assert_allclose(result_iterative.scores, result_analytical, atol=1e-6))

None


In [5]:
# Iterative result (has full metadata)                                                             
print("=== Iterative ===")                                                                         
print(f"Converged:  {result_iterative.converged}")                                                 
print(f"Iterations: {result_iterative.iterations}")                                                
print(f"Time:       {result_iterative.elapsed_sec:.2f}s")                                        
print(f"Score sum:  {result_iterative.scores.sum():.6f}")                                          
print(f"Top 5 nodes: {result_iterative.top_k(5, idx_to_id=loader.idx_to_id)}")                     
                                                                                                    
# Analytical result (plain ndarray — no metadata)                                                  
print("\n=== Analytical ===")                                                                      
print(f"Score sum:  {result_analytical.sum():.6f}")                                              
top5_idx = np.argsort(-result_analytical)[:5]                                                      
top5 = [(loader.idx_to_id[i], result_analytical[i]) for i in top5_idx]
print(f"Top 5 nodes: {top5}")                                                                      
                                  

=== Iterative ===
Converged:  True
Iterations: 16
Time:       0.00s
Score sum:  1.000000
Top 5 nodes: [(5, 0.24809927911889615), (4, 0.1855611425640593), (0, 0.1553362336832827), (3, 0.1422348653213814), (1, 0.13438423965619006)]

=== Analytical ===
Score sum:  1.000000
Top 5 nodes: [(5, 0.24809927820873667), (4, 0.1855611430373054), (0, 0.155336234156516), (3, 0.1422348653213619), (2, 0.13438423963804)]


In [7]:
P_VALUES = list(np.linspace(0.01, 0.99, 10))      
results = engine.run_sweep(P_VALUES)
top_k = 5

print(f"\n{'p':>6}  {'iters':>6}  {'max score':>10}  {'std':>10}  top-{top_k} nodes")
print("-" * 70)
for p, result in results.items():
    top = result.top_k(top_k, idx_to_id=loader.idx_to_id)
    top_str = "  ".join(f"{node}:{score:.4f}" for node, score in top)
    print(
        f"{p:>6.2f}  {result.iterations:>6}  "
        f"{result.scores.max():>10.4f}  {result.scores.std():>10.4f}  {top_str}"
    )




     p   iters   max score         std  top-5 nodes
----------------------------------------------------------------------
  0.01      19      0.2639      0.0476  5:0.2639  4:0.1864  0:0.1522  3:0.1382  1:0.1296
  0.12      17      0.2516      0.0420  5:0.2516  4:0.1858  0:0.1547  3:0.1413  1:0.1333
  0.23      15      0.2395      0.0364  5:0.2395  4:0.1849  0:0.1570  3:0.1445  1:0.1371
  0.34      14      0.2276      0.0309  5:0.2276  4:0.1837  0:0.1592  3:0.1476  1:0.1410
  0.45      12      0.2161      0.0255  5:0.2161  4:0.1821  0:0.1612  3:0.1508  1:0.1449
  0.55      11      0.2050      0.0202  5:0.2050  4:0.1801  0:0.1629  3:0.1540  1:0.1490
  0.66       9      0.1945      0.0150  5:0.1945  4:0.1777  0:0.1644  3:0.1571  1:0.1531
  0.77       8      0.1846      0.0100  5:0.1846  4:0.1747  0:0.1656  3:0.1602  1:0.1574
  0.88       6      0.1755      0.0051  5:0.1755  4:0.1712  0:0.1664  3:0.1633  1:0.1618
  0.99       4      0.1674      0.0004  5:0.1674  4:0.1671  0:0.1667  3:0.1